In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/prompt/collm_movie.txt
/kaggle/input/movie-lens/valid_ood2.pkl
/kaggle/input/movie-lens/test_ood2.pkl
/kaggle/input/movie-lens/train_ood2.pkl
/kaggle/input/movie-lens/valid_small_ood2.pkl
/kaggle/input/prompts/tallrec_movie.txt


In [2]:
!pip3 install peft
!pip3 install transformers
!pip3 install einops deepspeed
!pip3 install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 9.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 464.1/464.1 kB 26.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.24.7
    Uninstalling huggingface-hub-0.24.7:
      Successfully uninstalled huggingface-hub-0.24.7
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.4 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.8 MB/s eta 0:00:00
  Created wheel for deepspeed: filename=deepspeed-0.16.3-py3-none-any.whl size=1550054 sha256=e4d0f17cfe3c11bf3a2b1fcca5efb929dd0402840f8cced42a0bd74b69fd06d4
  Stored in directory: /root/.cache/pip/wheels/ca/e2/8f/3a91068b57481b104c9c450a20239ec874f6141f8b3769e0dd
Successfully built deepspeed
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 MB 25.0 

##  Import Required Libraries

In [3]:
import torch.nn as nn
from types import SimpleNamespace

from omegaconf import OmegaConf

import torch
from torch.utils.data.dataloader import default_collate, DataLoader
from torch.utils.data import Dataset, ConcatDataset
import torch.backends.cudnn as cudnn
from torch.nn import DataParallel

from peft import (
    prepare_model_for_kbit_training,
    get_peft_model_state_dict,
    get_peft_model,
    LoraConfig,
    PeftModel, 
    PeftConfig
)

import transformers
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    GenerationConfig,
    BitsAndBytesConfig
)

import random
import time
import math

from collections import defaultdict
from collections import deque

import datetime

from sklearn.metrics import (
            roc_auc_score,
            f1_score,
            precision_score,
            recall_score,
            log_loss,
        )

from itertools import cycle

import logging

import json

#======================================================================
# import accelerate
import accelerate
from accelerate import Accelerator
from accelerate.utils import set_seed
#======================================================================

## Parallel GPU Programming

In [4]:
# import os
# from accelerate.utils import write_basic_config
# write_basic_config() # Write a config file
# os._exit(0) # Restart the notebook to reload info from the latest config file 

In [5]:
# %load /root/.cache/huggingface/accelerate/default_config.yaml
"""
{
  "compute_environment": "LOCAL_MACHINE",
  "debug": false,
  "distributed_type": "MULTI_GPU",
  "downcast_bf16": false,
  "enable_cpu_affinity": false,
  "machine_rank": 0,
  "main_training_function": "main",
  "mixed_precision": "no",
  "num_machines": 1,
  "num_processes": 2,
  "rdzv_backend": "static",
  "same_network": false,
  "tpu_use_cluster": false,
  "tpu_use_sudo": false,
  "use_cpu": false
}
"""

'\n{\n  "compute_environment": "LOCAL_MACHINE",\n  "debug": false,\n  "distributed_type": "MULTI_GPU",\n  "downcast_bf16": false,\n  "enable_cpu_affinity": false,\n  "machine_rank": 0,\n  "main_training_function": "main",\n  "mixed_precision": "no",\n  "num_machines": 1,\n  "num_processes": 2,\n  "rdzv_backend": "static",\n  "same_network": false,\n  "tpu_use_cluster": false,\n  "tpu_use_sudo": false,\n  "use_cpu": false\n}\n'

## Set Up Config

In [6]:
# model
# TODO: improve the hardcoding config
rec_cfg = SimpleNamespace(
    user_num=839,
    item_num=3256,
    embedding_size=256
)

bnb_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# prompt_path = "/kaggle/input/prompts/tallrec_movie.txt"
prompt_path = "/kaggle/input/prompt/collm_movie.txt"
json_log_path = "/kaggle/working/metrics_log.json"
checkpoint_dir="/kaggle/working/"

training_cfg = SimpleNamespace(
    init_lr=1e-3,
    min_lr=8e-5,
    warmup_lr=1e-5,
    weight_decay=1e-3, # by default
    max_epoch=300,
    iters_per_epoch=100, #100 
    training_iters_per_epoch=50,
    eval_iters_per_epoch=50,
    batch_size_train=16, # 8
    batch_size_eval=64, # 32
    num_workers=2,
    warmup_steps=200,
    seed=42,
    # output_dir: ./log #log and model saving path
    
    amp=True,
    resume_ckpt_path=None,
    
    decay_rate=1,
    log_freq=50,
    accum_grad_iters=1,

    beta2 = 0.999,
    print_freq = 10,
    early_stopper_threshold = 20,
    ref_metric = 'auc',
    save_frequency = 2,
    mixed_precision= "no",
)

### Set up Hyperparameters

In [7]:
LORA_R = 8
LORA_ALPHA = 16
TARGET_MODULES = [
        "q_proj",
        "v_proj",
    ]
LORA_DROPOUT = 0.05

## Global Utils 

In [8]:
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

device = get_device()

In [9]:
## TODO: add this part of code in future
def get_dataset_path(dataset_name, stage):
    return "/kaggle/input/" + dataset_name + "/" + stage + "_ood2.pkl"

In [10]:
def move_sample_to_cuda(sample):
    return {key: value.to("cuda") if torch.is_tensor(value) else value for key, value in sample.items()}

In [11]:
def compute_user_auc(user, predict, label):
    """
    Compute user-level AUC for the given predictions and labels.

    Args:
        user (np.ndarray): User IDs associated with each prediction.
        predict (np.ndarray): Predicted scores.
        label (np.ndarray): True binary labels (0 or 1).

    Returns:
        float: The overall user-level AUC (uAUC).
        list: List of users with valid AUC scores.
        np.ndarray: AUC scores for individual users.
    """
    predict = predict.squeeze()
    label = label.squeeze()

    start_time = time.time()

    # Group user interactions
    users, inverse_indices, counts = np.unique(user, return_inverse=True, return_counts=True)
    sorted_indices = np.argsort(inverse_indices)

    # Create dictionary for user interactions
    candidates_dict = {}
    total_num = 0
    only_one_interaction = 0

    for idx, user_id in enumerate(users):
        start, end = total_num, total_num + counts[idx]
        user_indices = sorted_indices[start:end]

        if counts[idx] == 1:
            only_one_interaction += 1
        else:
            candidates_dict[user_id] = (predict[user_indices], label[user_indices])

        total_num += counts[idx]

    print(f"Users with only one interaction: {only_one_interaction}")

    # Compute AUC for each user
    auc_scores = []
    valid_users = []
    invalid_users = 0

    for user_id, (user_preds, user_labels) in candidates_dict.items():
        try:
            user_auc = roc_auc_score(user_labels, user_preds)
            auc_scores.append(user_auc)
            valid_users.append(user_id)
        except ValueError:  # Handle cases where labels are only one class
            invalid_users += 1

    auc_scores = np.array(auc_scores)

    print(f"Valid users: {len(valid_users)}, Invalid users: {invalid_users}")

    # Compute overall uAUC
    uauc = auc_scores.mean() if len(auc_scores) > 0 else 0.0
    elapsed_time = time.time() - start_time
    # print(f"uAUC computation time: {elapsed_time:.2f}s, uAUC: {uauc:.4f}")

    return uauc, valid_users, auc_scores

### SmoothedValue

In [12]:
class SmoothedValue(object):
    """Track a series of values and provide access to smoothed values over a
    window or the global series average.
    """

    def __init__(self, window_size=20, fmt=None):
        if fmt is None:
            fmt = "{median:.4f} ({global_avg:.4f})"
        self.deque = deque(maxlen=window_size)
        self.total = 0.0
        self.count = 0
        self.fmt = fmt

    def update(self, value, n=1):
        self.deque.append(value)
        self.count += n
        self.total += value * n

    @property
    def median(self):
        d = torch.tensor(list(self.deque))
        return d.median().item()

    @property
    def avg(self):
        d = torch.tensor(list(self.deque), dtype=torch.float32)
        return d.mean().item()

    @property
    def global_avg(self):
        return self.total / self.count

    @property
    def max(self):
        return max(self.deque)

    @property
    def value(self):
        return self.deque[-1]

    def __str__(self):
        return self.fmt.format(
            median=self.median,
            avg=self.avg,
            global_avg=self.global_avg,
            max=self.max,
            value=self.value,
        )

### Logger

In [13]:
class MetricLogger(object):
    def __init__(self, delimiter="\t"):
        self.meters = defaultdict(SmoothedValue)
        self.delimiter = delimiter

    def update(self, **kwargs):
        for k, v in kwargs.items():
            if isinstance(v, torch.Tensor):
                v = v.item()
            assert isinstance(v, (float, int))
            self.meters[k].update(v)

    def __getattr__(self, attr):
        if attr in self.meters:
            return self.meters[attr]
        if attr in self.__dict__:
            return self.__dict__[attr]
        raise AttributeError(
            "'{}' object has no attribute '{}'".format(type(self).__name__, attr)
        )

    def __str__(self):
        loss_str = []
        for name, meter in self.meters.items():
            loss_str.append("{}: {}".format(name, str(meter)))
        return self.delimiter.join(loss_str)

    def global_avg(self):
        loss_str = []
        for name, meter in self.meters.items():
            loss_str.append("{}: {:.6f}".format(name, meter.global_avg))
        return self.delimiter.join(loss_str)

    def add_meter(self, name, meter):
        self.meters[name] = meter

    def log_every(self, iterable, print_freq, header=None):
        i = 0
        if not header:
            header = ""
        start_time = time.time()
        end = time.time()
        iter_time = SmoothedValue(fmt="{avg:.4f}")
        data_time = SmoothedValue(fmt="{avg:.4f}")
        space_fmt = ":" + str(len(str(len(iterable)))) + "d"
        log_msg = [
            header,
            "[{0" + space_fmt + "}/{1}]",
            "eta: {eta}",
            "{meters}",
            "time: {time}",
            "data: {data}",
        ]
        if torch.cuda.is_available():
            log_msg.append("max mem: {memory:.0f}")
        log_msg = self.delimiter.join(log_msg)
        MB = 1024.0 * 1024.0
        for obj in iterable:
            data_time.update(time.time() - end)
            yield obj
            iter_time.update(time.time() - end)
            if i % print_freq == 0 or i == len(iterable) - 1:
                eta_seconds = iter_time.global_avg * (len(iterable) - i)
                eta_string = str(datetime.timedelta(seconds=int(eta_seconds)))
                if torch.cuda.is_available():
                    print(
                        log_msg.format(
                            i,
                            len(iterable),
                            eta=eta_string,
                            meters=str(self),
                            time=str(iter_time),
                            data=str(data_time),
                            memory=torch.cuda.max_memory_allocated() / MB,
                        )
                    )
                else:
                    print(
                        log_msg.format(
                            i,
                            len(iterable),
                            eta=eta_string,
                            meters=str(self),
                            time=str(iter_time),
                            data=str(data_time),
                        )
                    )
            i += 1
            end = time.time()
        total_time = time.time() - start_time
        total_time_str = str(datetime.timedelta(seconds=int(total_time)))
        print(
            "{} Total time: {} ({:.4f} s / it)".format(
                header, total_time_str, total_time / len(iterable)
            )
        )

### LRScheduler

In [14]:
# CosineAnnealing	For restart or cyclical schedules	Good for fine-tuning
class LinearWarmupCosineLRScheduler:
    def __init__(
        self,
        optimizer,
        max_epoch,
        iters_per_epoch,
        min_lr,
        init_lr,
        warmup_steps,
        warmup_start_lr=-1,
        **kwargs
    ):
        self.optimizer = optimizer

        self.max_epoch = max_epoch
        self.iters_per_epoch = iters_per_epoch
        self.min_lr = min_lr

        self.init_lr = init_lr
        self.warmup_steps = warmup_steps
        self.warmup_start_lr = warmup_start_lr if warmup_start_lr >= 0 else init_lr

    def step(self, cur_epoch, cur_step):
        total_cur_step = cur_epoch * self.iters_per_epoch + cur_step
        if total_cur_step < self.warmup_steps:
            warmup_lr_schedule(
                step=cur_step,
                optimizer=self.optimizer,
                max_step=self.warmup_steps,
                init_lr=self.warmup_start_lr,
                max_lr=self.init_lr,
            )
        else:
            cosine_lr_schedule(
                epoch=total_cur_step,
                optimizer=self.optimizer,
                max_epoch=self.max_epoch * self.iters_per_epoch,
                init_lr=self.init_lr,
                min_lr=self.min_lr,
            )


def cosine_lr_schedule(optimizer, epoch, max_epoch, init_lr, min_lr):
    """Decay the learning rate"""
    lr = (init_lr - min_lr) * 0.5 * (
        1.0 + math.cos(math.pi * epoch / max_epoch)
    ) + min_lr
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr


def warmup_lr_schedule(optimizer, step, max_step, init_lr, max_lr):
    """Warmup the learning rate"""
    lr = min(max_lr, init_lr + (max_lr - init_lr) * step / max(max_step, 1))
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

### Early Stopper

In [15]:
class EarlyStopper:
    """Monitor a metric and stop training if no improvement after a given patience."""
    def __init__(self, ref_metric='valid_auc', increase=True, patience=20):
        self.ref_metric = ref_metric
        self.increase = increase
        self.patience = patience
        self.best_metric = None
        self.reach_count = 0

    def update(self, metrics):
        current_value = metrics[self.ref_metric]
        if self.best_metric is None or (self.increase and current_value > self.best_metric[self.ref_metric]) or \
                (not self.increase and current_value < self.best_metric[self.ref_metric]):
            self.best_metric = metrics
            self.reach_count = 0
            return True
        else:
            self.reach_count += 1
            return False

    def should_stop(self):
        return self.reach_count >= self.patience

## Prepare training samples

### Process datasets

In [16]:
class BaseProcessor:
    def __init__(self):
        self.transform = self.linear
        return

    def linear(self, x):
        return x

    def __call__(self, item):
        return self.transform(item)

    @classmethod
    def from_config(cls, cfg=None):
        return cls()

    def build(self, **kwargs):
        cfg = OmegaConf.create(kwargs)
        return self.from_config(cfg)

In [17]:
class RecBaseDataset(Dataset):
    def __init__(
        self, text_processor=None, stage=None
    ):
        self.annotation = pd.read_pickle(get_dataset_path("movie-lens", stage)).values
        self.text_processor = text_processor

    def __len__(self):
        return len(self.annotation)

    def collater(self, samples):
        return default_collate(samples)

    def set_processors(self, text_processor):
        self.text_processor = text_processor

    def _add_instance_ids(self, key="instance_id"):
        for idx, ann in enumerate(self.annotation):
            ann[key] = str(idx)

In [18]:
def convert_title_list(titles):
    """
    Convert a list of titles into a single string where each title is wrapped in quotes and separated by commas.
    If the list is empty or contains no valid titles, return "unkow".

    Args:
        titles (list): List of title strings.

    Returns:
        str: A formatted string of quoted titles or "unkow" if no valid titles are present.
    """
    # Filter titles with non-zero length and wrap in quotes
    quoted_titles = [f'"{title}"' for title in titles if title]

    # Return the joined string or "unkow" if the list is empty
    return ", ".join(quoted_titles) if quoted_titles else "unkow"

In [19]:
class MoiveOOData(RecBaseDataset):
    def __init__(self, text_processor=None, stage=None):
        super().__init__(text_processor=text_processor, stage=stage)
        self.annotation = pd.read_pickle(get_dataset_path("movie-lens", stage)).reset_index(drop=True)

        self.use_his = False
        self.prompt_flag = False

        if 'sessionItems' in self.annotation.columns or 'his' in self.annotation.columns:
            used_columns = ['uid','iid','title','his', 'his_title','label']
            renamed_columns = ['UserID','TargetItemID','TargetItemTitle', 'InteractedItemIDs', 'InteractedItemTitles','label']
            if 'not_cold' in self.annotation.columns:
                used_columns.append("not_cold")
                renamed_columns.append("prompt_flag")
                self.prompt_flag = True

            self.use_his = True
            self.annotation = self.annotation[used_columns]
            self.annotation.columns = renamed_columns
        
            self.annotation["InteractedItemIDs"] = self.annotation["InteractedItemIDs"].map(list)
            self.annotation["InteractedItemTitles"] = self.annotation["InteractedItemTitles"].map(list)
            self.annotation["InteractedItemTitles"] = self.annotation["InteractedItemTitles"] #.map(convert_title_list)
        else:
            used_columns = ['uid','iid','title','label']
            renamed_columns = ['UserID','TargetItemID','TargetItemTitle','label']
            if 'not_cold' in self.annotation.columns:
                used_columns.append('not_cold')
                renamed_columns.append("prompt_flag")
                self.prompt_flag = True
            self.annotation = self.annotation[used_columns]
            self.annotation.columns = renamed_columns
        
        # print("data path:", ann_paths[0], "data size:", self.annotation.shape)
        self.user_num = self.annotation['UserID'].max()+1
        self.item_num = self.annotation['TargetItemID'].max()+1
        self.text_processor = text_processor
        
        if self.use_his:
            max_length_ = 0
            for x in self.annotation['InteractedItemIDs'].values:
                max_length_ = max(max_length_, len(x))
            self.max_lenght = min(max_length_, 10) # average: only 50; 0915: 15 
            print("Movie OOD datasets, max history length:", self.max_lenght)
            
    def __getitem__(self, index):
        # TODO this assumes image input, not general enough
        ann = self.annotation.iloc[index]
        if self.use_his:
            a = ann['InteractedItemIDs']
            InteractedNum = len(a)
            if a[0] == 0:
                InteractedNum -= 1

            if len(a) < self.max_lenght:
                b = [0]* (self.max_lenght-len(a)) # assuming padding idx is zero
                b.extend(a)
            elif len(a)> self.max_lenght:
                b = a[-self.max_lenght:]
                InteractedNum = self.max_lenght
            else:
                b = a
            one_sample = {
                "UserID": ann['UserID'],
                "InteractedItemIDs_pad": np.array(b),
                "InteractedItemTitles": convert_title_list(ann['InteractedItemTitles'][-InteractedNum:]),
                "TargetItemID": ann["TargetItemID"],
                "TargetItemTitle": "\""+ann["TargetItemTitle"].strip(' ')+"\"",
                "InteractedNum": InteractedNum,
                "label": ann['label']
            }
            if self.prompt_flag:
                one_sample['prompt_flag'] = ann['prompt_flag']
            return one_sample 
        else:
            one_sample = {
                "UserID": ann['UserID'],
                "TargetItemID": ann["TargetItemID"],
                "TargetItemTitle": ann["TargetItemTitle"].strip(' '),
                "label": ann['label']
            }
            if self.prompt_flag:
                one_sample['prompt_flag'] = ann['prompt_flag']
            return one_sample 

In [20]:
def build_datasets():
    """
    Builds the train, validation, and test datasets using a shared text processor.
    All annotations and media are expected to be downloaded to the specified locations.
    """

    # Initialize shared components
    text_processor = BaseProcessor()
    datasets = {}

    # Define dataset keys
    dataset_keys = ['train', 'valid', 'test']

    # Build datasets dynamically
    for key_ in dataset_keys:
        datasets[key_] = MoiveOOData(text_processor=text_processor, stage=key_)

    return datasets

In [21]:
datasets = build_datasets()

Movie OOD datasets, max history length: 10
Movie OOD datasets, max history length: 10
Movie OOD datasets, max history length: 10


In [22]:
print(len(datasets["train"])) # 33891
print(len(datasets["valid"])) # 10401
print(len(datasets["test"])) # 7331

33891
10401
7331


### Process Dataloaders

In [23]:
def create_dataloaders(datasets):
    """
    Creates a dictionary of DataLoaders for the given datasets.
    
    Args:
        datasets (dict): A dictionary where keys are dataset split names (e.g., "train", "val")
                         and values are the corresponding dataset objects.
    
    Returns:
        dict: A dictionary where keys are the split names and values are the DataLoaders.
    """
    def _create_dataloader(dataset, batch_size, num_workers):
        """
        Creates a DataLoader for a specific dataset.
        
        Args:
            dataset: The dataset object.
            batch_size (int): Batch size for the DataLoader.
            num_workers (int): Number of worker threads for data loading.
        
        Returns:
            iter: An iterator over the DataLoader.
        """
        data_loader = DataLoader(
            dataset,
            batch_size=batch_size,
            num_workers=num_workers,
            pin_memory=True,
        )
        if not hasattr(data_loader, "__next__"):
            # Convert to iterator if not already
            data_loader = iter(data_loader)
        return cycle(data_loader)
    
    # Use dictionary comprehension to simplify logic
    data_loaders = {
        split_name: _create_dataloader(
            dataset,
            batch_size=training_cfg.batch_size_train if split_name == "train" else training_cfg.batch_size_eval,
            num_workers=training_cfg.num_workers
        )
        for split_name, dataset in datasets.items()
    }

    return data_loaders

## Define the Rec Model 

In [24]:
# TODO: add device config
class MatrixFactorization(nn.Module):
    # here we does not consider the bais term 
    def __init__(self, rec_config, *args, **kwargs) -> None:
        super().__init__()
        self.padding_index = 0
        self.rec_config = rec_config
        self.device = device
        self.user_embedding = nn.Embedding(self.rec_config.user_num, self.rec_config.embedding_size, padding_idx=self.padding_index).to(self.device)
        self.item_embedding = nn.Embedding(self.rec_config.item_num, self.rec_config.embedding_size, padding_idx=self.padding_index).to(self.device)
        print("creat MF model, user num:", self.rec_config.user_num, "item num:", self.rec_config.item_num)

    def user_encoder(self,users,all_users=None):
        # print("user max:", users.max(), users.min())
        return self.user_embedding(users)
        
    def item_encoder(self,items,all_items=None):
        # print("items max:", items.max(), items.min())
        return self.item_embedding(items)
    
    def computer(self): # does not need to compute user reprensentation, directly taking the embedding as user/item representations
        return None, None
    
    def forward(self,users,items):
        user_embedding = self.user_embedding(users)
        item_embedding = self.item_embedding(items)
        matching = torch.mul(user_embedding, item_embedding).sum(dim=-1)
        return matching

In [25]:
def get_ids_order(prompt):
    """
    Determines the order of ID placeholders in the given prompt.

    Args:
        prompt (str): The input string containing placeholders like "<UserID>", "<ItemIDList>", "<TargetItemID>".

    Returns:
        list: A list of indices indicating the order of ID placeholders based on their positions in the prompt.
    """
    id_flags = ["<UserID>", "<ItemIDList>", "<TargetItemID>"]

    # Find the positions of ID flags in the prompt
    positions = [prompt.find(flag) for flag in id_flags if flag in prompt]

    # Return the order of indices sorted by positions
    return np.argsort(positions).tolist()

In [26]:
class Model(nn.Module):

    def __init__(self, rec_config=rec_cfg, proj_mid=5, proj_token_num=1, max_txt_len=1024):
        super().__init__()
        self.proj_token_num = proj_token_num

        print("Loading Device Type")
        self.device = device     
        print("Loading Device Type Done")

        print("Loading the Rec Model")
        # TODO: implement load from pretrained
        # TODO: implement freeze
        self.rec_encoder = MatrixFactorization(rec_config).to(self.device)
        print("Loading Rec Model Done")

        print("Loading Llama")
        # self.llama_model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map={"":0})
        self.llama_model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)
        self.llama_tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        # self.llama_tokenizer.pad_token = (0)  # unk. we want this to be different from the eos token
        
        self.llama_tokenizer.pad_token = self.llama_tokenizer.eos_token
        self.llama_tokenizer.padding_side = "left"  # Allow batched inference

        print("Set up Lora Config")
        lora_config = LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            target_modules=TARGET_MODULES,
            lora_dropout=LORA_DROPOUT,
            bias="none",
            task_type="CAUSAL_LM",
        )
        
        self.llama_model_lora = get_peft_model(self.llama_model, lora_config)
        print("Setting Lora Done")
        print("Loading Llama Done")

        print("Loading Mapping Layer")
        self.llama_proj = nn.Sequential(
            nn.Linear(self.rec_encoder.rec_config.embedding_size, self.rec_encoder.rec_config.embedding_size*int(proj_mid)),  # ml100=>5
            nn.ReLU(),
            nn.Linear(self.rec_encoder.rec_config.embedding_size*int(proj_mid), self.llama_model.config.hidden_size * self.proj_token_num),
        ).to(self.device)

        print("Loading Mapping Layer Done")

        print("Loading Prompts")
        self.load_prompt(prompt_path)
        print("Loading Prompts Done")

        print("Set other attributes")
        self.max_txt_len = max_txt_len

        print("Set answer type")
        self.pos_ans = ['Yes']
        self.neg_ans = ['No']
        print("Set answer type done")


    def load_prompt(self, prompt_path="", prompt_template="{}"):
        """
        Load and process prompts from a specified file path.
    
        Args:
            prompt_path (str): Path to the file containing raw prompts. 
                               If empty, no prompts are loaded.
    
        Sets:
            self.prompt_list: List of formatted prompts.
            self.prompt_list_p: Placeholder for additional processing (None by default).
        """
        if not prompt_path:
            self.prompt_list = []
            self.prompt_list_p = None
            return

        try:
            # Load prompts from the specified file
            with open(prompt_path, 'r') as f:
                raw_prompts = f.read().splitlines()
    
            # Filter and format prompts
            filtered_prompts = [raw_prompt for raw_prompt in raw_prompts]
            self.prompt_list = [prompt_template.format(p) for p in filtered_prompts]
    
            # Log prompt loading
            print(f"Loaded {len(self.prompt_list)} training prompts")
            print("Prompt List:\n", "\n".join(self.prompt_list))
    
            # Initialize additional attributes
            self.prompt_list_p = None
        except FileNotFoundError:
            print(f"Error: The file at '{prompt_path}' was not found.")
            self.prompt_list = []
            self.prompt_list_p = None


    def prompt_with_p(self, p):
        """
        Generates or retrieves the weighted prompt list based on probabilities `p`.
    
        Args:
            p (list): A list of integers where each value indicates how many times 
                      to repeat the corresponding prompt in `self.prompt_list`.
    
        Returns:
            list: A weighted list of prompts (`self.prompt_list_p`) where prompts 
                  are repeated based on the input list `p`.
        """
        if self.prompt_list_p is not None:
            # If the weighted prompt list already exists, return it
            return self.prompt_list_p
    
        # Ensure `self.prompt_list` exists and matches the length of `p`
        if not self.prompt_list or len(self.prompt_list) != len(p):
            raise ValueError("The length of `p` must match `self.prompt_list`.")
    
        # Generate a weighted prompt list
        prompt_list_p = []
        for k, repeat_count in enumerate(p):
            prompt_list_p.extend([self.prompt_list[k]] * repeat_count)
    
        # Cache the generated prompt list for reuse
        self.prompt_list_p = prompt_list_p
        return self.prompt_list_p


    def maybe_autocast(self, dtype=torch.float16):
        """
        Returns the appropriate autocast context manager based on the device.
    
        Args:
            dtype (torch.dtype): The data type to use for autocast (default: torch.float16).
    
        Returns:
            Context manager: An autocast context manager for GPU, or a no-op context manager for CPU.
        """
        # Use autocast only if the device is not CPU
        if self.device.type == "cuda":
            return torch.amp.autocast('cuda', dtype=dtype)
        return nullcontext()


    def encode_recdata(self, sample, ids_order=None):  # used for stage2    
        with self.maybe_autocast():
            batch_size = sample['UserID'].shape[0]
            hidden_size = self.llama_model.config.hidden_size
            all_user_embeds, all_item_embeds = self.rec_encoder.computer()

            user_embeds = self.rec_encoder.user_encoder(sample['UserID'], all_users=all_user_embeds).unsqueeze(-2)
            targetItem_embed = self.rec_encoder.item_encoder(sample['TargetItemID'], all_items=all_item_embeds).unsqueeze(-2)
            
            user_embeds_llama = self.llama_proj(user_embeds).reshape(batch_size, -1, self.proj_token_num, hidden_size)

            targetItem_embeds_llama = self.llama_proj(targetItem_embed).reshape(batch_size, -1, self.proj_token_num, hidden_size)
            
            # loss_c = consitence_loss(user_embeds, user_embeds_llama) + consitence_loss(targetItem_embed, targetItem_embeds_llama)
            # ATTENTION This part has been changed, used to be 3
            if 'InteractedItemIDs_pad' in sample.keys() and len(ids_order)==2:
                interactedItem_embeds = self.rec_encoder.item_encoder(sample['InteractedItemIDs_pad'], all_items=all_item_embeds)
                interactedItem_embeds_llama = self.llama_proj(interactedItem_embeds).reshape(batch_size,-1, self.proj_token_num, hidden_size)

                merged_embeds = [user_embeds_llama, interactedItem_embeds_llama, targetItem_embeds_llama]
                merged_embeds = [merged_embeds[k] for k in ids_order]
                merged_embeds = torch.cat(merged_embeds,dim=1)              
                idx_flag = torch.ones_like(sample['InteractedItemIDs_pad'])
                idx_flag = torch.where(sample['InteractedItemIDs_pad']==self.rec_encoder.padding_index, 0, idx_flag) # indx_of_paddded historical items
                # to indicate user_id, his_items_id, target_item_id
                idx_flag = [torch.ones([idx_flag.shape[0],1]).to(idx_flag.device),idx_flag,torch.ones([idx_flag.shape[0],1]).to(idx_flag.device)]
                idx_flag = [idx_flag[k] for k in ids_order]
                idx_flag = torch.cat(idx_flag,dim=1).to(device)
                idx_nopad = torch.nonzero(idx_flag)               

                sample_embeds_llama = {
                    'User_emb': user_embeds_llama.reshape(batch_size,-1, hidden_size),
                    'TargetItem_emb': targetItem_embeds_llama.reshape(batch_size,-1, hidden_size),
                    'InteractedItems_embs': interactedItem_embeds_llama.reshape(batch_size,-1, hidden_size),
                    'merged_embs': merged_embeds[idx_nopad[:,0],idx_nopad[:,1]].reshape(-1, hidden_size),
                    # 'loss_c': loss_c
                }
            else:
                sample_embeds_llama = {
                    'User_emb': user_embeds_llama.reshape(batch_size,-1, hidden_size),
                    'TargetItem_emb': targetItem_embeds_llama.reshape(batch_size,-1, hidden_size),
                    'InteractedItems_embs': None,
                    'merged_embs': None,
                    # 'loss_c': loss_c
                }
        sample_atts_llama = None

        return sample_embeds_llama, sample_atts_llama


    # TODO: refactoring 
    def recprompt_wrap(self, samples, ori_samples, atts_sample, prompt): # used for stage 2
        if prompt:
            prompt_ori = prompt
            split_symbol = ["<UserID>", "<ItemIDList>", "<ItemTitleList>", "<TargetItemID>", "<TargetItemTitle>"]
            batch_size = ori_samples['UserID'].shape[0]
            bos = "<s>"
            unk_ = self.llama_tokenizer.unk_token #"<unk>"
            unk_ = ".".join([unk_]*self.proj_token_num)
            prompt = bos + prompt # add the bos

            prompt = prompt.replace("<UserID>", unk_)
            prompt = prompt.replace("<TargetItemID>", unk_)

            # interactedItems = samples['InteractedItemTitles']
            prompt_list = []
            
            for k in range(batch_size):
                prompt_ = prompt+""
                if 'InteractedNum' in ori_samples.keys():
                    prompt_ = prompt_.replace('<ItemIDList>', ', '.join([unk_]*ori_samples['InteractedNum'][k]))
                    prompt_ = prompt_.replace("<ItemTitleList>", ori_samples['InteractedItemTitles'][k])
                prompt_ = prompt_.replace("<TargetItemTitle>", ori_samples['TargetItemTitle'][k])
                prompt_list.append(prompt_)
            
            self.llama_tokenizer.padding_side = "left"
            prompts_tokens = self.llama_tokenizer(
                prompt_list,
                return_tensors="pt",
                padding="longest",
                truncation=True,
                max_length=self.max_txt_len,
                add_special_tokens=False
            ).to(ori_samples['UserID'].device)
            unk_token_id = self.llama_tokenizer.unk_token_id
            # print("#######prmpt decoded example: ",' '.join(self.llama_tokenizer.batch_decode(prompts_tokens.input_ids[0])))
                
            replaced_idx = torch.nonzero(prompts_tokens.input_ids==unk_token_id)
            prompt_embeds = self.llama_model.model.embed_tokens(prompts_tokens.input_ids)
 
            if "<UserID>" in prompt_ori  and "<ItemIDList>" in prompt_ori and  "<TargetItemID>" in prompt_ori:
                prompt_embeds[replaced_idx[:,0],replaced_idx[:,1]] = samples['merged_embs']
            elif "<UserID>" in prompt_ori and "<TargetItemID>" in prompt_ori and "<ItemIDList>" not in prompt_ori:       
                # Prepare value tensor
                value_tensor = torch.cat([samples['User_emb'], samples['TargetItem_emb']], dim=-2)
                value_tensor = value_tensor.reshape(-1, samples['User_emb'].shape[-1])
                
                # Ensure shapes match
                if replaced_idx.shape[0] != value_tensor.shape[0]:
                    print(f"Trimming value tensor to match replaced_idx: {replaced_idx.shape[0]} rows")
                    value_tensor = value_tensor[:replaced_idx.shape[0]]
                
                # Perform the update
                prompt_embeds[replaced_idx[:, 0], replaced_idx[:, 1]] = value_tensor
            else:
                pass 
            return prompt_embeds, prompts_tokens.attention_mask


    def prompt_based_encode(self,prompt, samples):
        id_orders = get_ids_order(prompt)
        samples_encode, atts_samples = self.encode_recdata(samples,ids_order=id_orders)
        sample_embeds, atts_samples = self.recprompt_wrap(samples_encode, samples, atts_samples, prompt)
        return sample_embeds, atts_samples
    

    def _prepare_prompt_based_encoding(self, samples, user_selective_prompts=False):
        """Prepare sample embeddings and attention masks based on the selected prompt."""
        if not self.prompt_list:
            return None, None

        if user_selective_prompts:
            prompt_flag = samples['prompt_flag']
            unique_flags = torch.unique(prompt_flag)
            sample_embeds, atts_samples, true_idx = [], [], torch.zeros_like(prompt_flag)
            pre_ = 0
            for k_flag in unique_flags:
                idx_k = torch.nonzero(prompt_flag == k_flag).squeeze()
                true_idx[idx_k] = pre_ + torch.arange(idx_k.shape[0])
                pre_ += idx_k.shape[0]
                sub_k_sample = {key: samples[key][idx_k] for key in samples.keys()}
                used_prompt = self.prompt_list[-1] if k_flag == 0 else self.prompt_list[1]
                sample_embeds_k, atts_samples_k = self.prompt_based_encode(used_prompt, sub_k_sample)
                sample_embeds.append(sample_embeds_k)
                atts_samples.append(atts_samples_k)
            sample_embeds = torch.cat(sample_embeds, dim=0)
            atts_samples = torch.cat(atts_samples, dim=0)
            sample_embeds = sample_embeds[true_idx]
            atts_samples = atts_samples[true_idx]
        else:
            prompt = random.choice(self.prompt_list) if isinstance(self.prompt_list, list) else self.prompt_list[0]
            sample_embeds, atts_samples = self.prompt_based_encode(prompt, samples)

        return sample_embeds, atts_samples


    def _prepare_tokenizer_inputs(self, samples):
        """Prepare tokenizer inputs and target tensors for regression."""
        ans_ = {1: self.pos_ans[0], 0: self.neg_ans[0]}
        text = [ans_[int(t)] for t in samples["label"]]

        to_regress_tokens = self.llama_tokenizer(
            text,
            return_tensors="pt",
            padding="longest",
            truncation=True,
            max_length=self.max_txt_len,
            add_special_tokens=False,
        ).to(self.device)

        targets = to_regress_tokens.input_ids.masked_fill(
            to_regress_tokens.input_ids == self.llama_tokenizer.pad_token_id, -100
        )
        return to_regress_tokens, targets

    
    def _shared_forward_generate(self, sample_embeds, atts_samples, to_regress_tokens, targets):
        """
        Shared logic for the forward and generate_for_samples methods:
        - Combines embeddings
        - Prepares attention masks
        - Passes data through the model
        """
        empty_targets = torch.ones(
            [atts_samples.shape[0], atts_samples.shape[1]], dtype=torch.long
        ).to(self.device).fill_(-100)

        targets = torch.cat([empty_targets, targets], dim=1)
        to_regress_embeds = self.llama_model.model.embed_tokens(to_regress_tokens.input_ids)
        inputs_embeds = torch.cat([sample_embeds, to_regress_embeds], dim=1)
        attention_mask = torch.cat([atts_samples, to_regress_tokens.attention_mask], dim=1)

        with self.maybe_autocast():
            outputs = self.llama_model_lora(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                return_dict=True,
                labels=targets,
            )

        pos_ans_id = self.llama_tokenizer(self.pos_ans[0], add_special_tokens=False).input_ids[0]
        logits = outputs.logits[:, -to_regress_tokens.input_ids.shape[-1] - 1, pos_ans_id]
        return outputs, logits

    
    def forward(self, samples):
        """
        Forward pass for training with loss computation.
        process inputs (encode data, wrap_prompt), prepare embeddings, calculate logits, and compute a loss.
        Used for training phase
        """
        sample_embeds, atts_samples = self._prepare_prompt_based_encoding(samples)
        self.llama_tokenizer.padding_side = "right"

        to_regress_tokens, targets = self._prepare_tokenizer_inputs(samples)
        _, logits = self._shared_forward_generate(sample_embeds, atts_samples, to_regress_tokens, targets)

        # Calculate binary cross-entropy loss
        loss = nn.functional.binary_cross_entropy_with_logits(logits, samples["label"].float())
        return {"loss": loss}

    def generate_for_samples(self, samples, return_all=False, user_selective_prompts=False):
        """
        Generate logits for given samples, optionally returning all outputs.
        Used for eval phase
        """
        sample_embeds, atts_samples = self._prepare_prompt_based_encoding(samples, user_selective_prompts)
        self.llama_tokenizer.padding_side = "right"

        to_regress_tokens, targets = self._prepare_tokenizer_inputs(samples)
        outputs, logits = self._shared_forward_generate(sample_embeds, atts_samples, to_regress_tokens, targets)

        # Calculate binary cross-entropy loss
        loss = nn.functional.binary_cross_entropy_with_logits(logits, samples["label"].float())

        if return_all:
            return {"loss": loss, "logits": logits}, outputs

        return {"loss": loss, "logits": logits}

In [27]:
model = Model()
# model = DataParallel(model)
# model = model.to('cuda')

Loading Device Type
Loading Device Type Done
Loading the Rec Model
creat MF model, user num: 839 item num: 3256
Loading Rec Model Done
Loading Llama


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now set to True since model is quantized.


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Set up Lora Config
Setting Lora Done
Loading Llama Done
Loading Mapping Layer
Loading Mapping Layer Done
Loading Prompts
Loaded 4 training prompts
Prompt List:
 #Question: A user has given high ratings to the following movies: <ItemTitleList>. Additionally, we have information about the user's preferences encoded in the feature <UserID>. Using all available information, make a prediction about whether the user would enjoy the movie titled <TargetItemTitle> with the feature <TargetItemID>? Answer with "Yes" or "No". \n#Answer:
#Question: A user has given high ratings to the following movies: <ItemTitleList>. Additionally, we have information about the user's preferences encoded in the feature <UserID>. Using all available information, make a prediction about whether the user would enjoy the movie titled <TargetItemTitle> with the feature <TargetItemID>? Answer with "Yes" or "No". \n#Answer:
#Question: A user has given high ratings to the following movies: <ItemTitleList>. Additionally, 

## Prepare the Runner

In [28]:
class Runner:
    """
    A runner class to train and evaluate a model given a task and datasets.

    The runner uses pytorch distributed data parallel by default. Future release
    will support other distributed frameworks.
    """

    def __init__(self, cfg, model, dataloaders):
        self.training_config = cfg

        self.dataloaders = dataloaders

        self._model = model

        self._device = None
        self._scaler = None
        self._dataloaders = None
        self.optimizer = self.get_optimizer()
        self._lr_sched = self.get_lr_sched()
        self._accelerator = None

        self.start_epoch = 0
        # TODO: check early_stopper works fine
        self.early_stopper = EarlyStopper(ref_metric=self.training_config.ref_metric, increase=True, patience=self.training_config.early_stopper_threshold)
        self.json_log_path = json_log_path

        self.train_loader = self.get_train_loader()
        self.test_loader = self.get_test_loader()
        


    def get_optimizer(self):
        num_parameters = 0
        p_wd, p_non_wd = [], []
        for n, p in self._model.named_parameters():
            if not p.requires_grad:
                continue  # frozen weights
            if p.ndim < 2 or "bias" in n or "ln" in n or "bn" in n:
                p_non_wd.append(p)
            else:
                p_wd.append(p)
            num_parameters += p.data.nelement()
        print("number of trainable parameters: %d" % num_parameters)
        self._num_trainable_para = num_parameters > 0
        optim_params = [
            {
                "params": p_wd,
                "weight_decay": float(self.training_config.weight_decay),
            },
            {"params": p_non_wd, "weight_decay": 0},
        ]

        beta2 = self.training_config.beta2
        init_lr = self.training_config.init_lr
        
        optimizer = torch.optim.AdamW(
            optim_params,
            lr=float(init_lr),
            weight_decay=float(self.training_config.weight_decay),
            betas=(0.9, beta2),
        )

        return optimizer

    @property
    def scaler(self):
        if self._scaler is None:
            self._scaler = torch.amp.GradScaler('cuda')
        return self._scaler


    def get_lr_sched(self):
        max_epoch = self.max_epoch
        min_lr = self.min_lr
        init_lr = self.init_lr

        # optional parameters
        decay_rate = self.training_config.decay_rate
        warmup_start_lr = self.training_config.warmup_lr
        warmup_steps = self.training_config.warmup_steps
        iters_per_epoch = self.training_config.iters_per_epoch


        lr_sched = LinearWarmupCosineLRScheduler(
            optimizer=self.optimizer,
            max_epoch=max_epoch,
            iters_per_epoch=iters_per_epoch,
            min_lr=min_lr,
            init_lr=init_lr,
            decay_rate=decay_rate,
            warmup_start_lr=warmup_start_lr,
            warmup_steps=warmup_steps,
        )

        return lr_sched

    @property
    def cuda_enabled(self):
        return True

    @property
    def max_epoch(self):
        return int(self.training_config.max_epoch)

    @property
    def log_freq(self):
        return int(self.training_config.log_freq)

    @property
    def init_lr(self):
        return float(self.training_config.init_lr)

    @property
    def min_lr(self):
        return float(self.training_config.min_lr)

    @property
    def accum_grad_iters(self):
        return int(self.training_config.accum_grad_iters)


    def get_train_loader(self):
        train_dataloader = self.dataloaders["train"]
        return train_dataloader


    def get_test_loader(self):
        test_dataloader = self.dataloaders["valid"]
        return test_dataloader

    @property
    def save_frequency(self):
        return int(self.training_config.save_frequency)

    @property
    def mixed_precision(self):
        return self.training_config.mixed_precision


    def prepare_accelerator(self):
        self._accelerator = Accelerator(mixed_precision=self.mixed_precision)
        self._accelerator.print(f'device {str(self._accelerator.device)} is used!')
        self._model, self.optimizer,self._lr_sched, self.train_loader, self.test_loader = self._accelerator.prepare(
        self._model, self.optimizer,self._lr_sched, self.train_loader, self.test_loader)
        print("prepare accelerator done")
        
        

    def train_epoch(self, epoch):
    # train
        self._model.train()
    
        return self._train_inner_loop(
            epoch=epoch,
            model=self._model,
            iters_per_epoch=self.training_config.training_iters_per_epoch,
            data_loader=self.train_loader,
            optimizer=self.optimizer,
            scaler=self.scaler,
            lr_scheduler=self._lr_sched,
            cuda_enabled=self.cuda_enabled,
            accum_grad_iters=self.accum_grad_iters,
        )


    # TODO: currently only implement epoch-based training
    def _train_inner_loop(
        self,
        epoch,
        iters_per_epoch,
        model,
        data_loader,
        optimizer,
        lr_scheduler,
        scaler=None,
        start_iters=None,
        cuda_enabled=False,
        accum_grad_iters=1,
    ):
        use_amp = scaler is not None
    
        metric_logger = MetricLogger(delimiter="  ")
        metric_logger.add_meter("lr", SmoothedValue(window_size=1, fmt="{value:.6f}"))
        metric_logger.add_meter("loss", SmoothedValue(window_size=1, fmt="{value:.4f}"))
    
        # if iter-based runner, schedule lr based on inner epoch.
        print(
            "Start training epoch {}, {} iters per inner epoch.".format(
                epoch, iters_per_epoch
            )
        )
        header = "Train: data epoch: [{}]".format(epoch)
    
        for i in metric_logger.log_every(range(iters_per_epoch), self.log_freq, header):
            # if using iter-based runner, we stop after iters_per_epoch iterations.
            if i >= iters_per_epoch:
                break

            samples = next(data_loader)
            
            samples.update(
                {
                    "epoch": epoch,
                    "num_iters_per_epoch": iters_per_epoch,
                    "iters": i, 
                }
            )
        
            samples = move_sample_to_cuda(samples)

            lr_scheduler.step(cur_epoch=epoch, cur_step=i)

            
            with torch.amp.autocast('cuda', enabled=use_amp):
            # with torch.cuda.amp.autocast(enabled=use_amp):
                loss = self.train_step(model=model, samples=samples)
            
            
            #======================================================================
            # attention here!
            """
            with self._accelerator.autocast():
                loss = self.train_step(model=model, samples=samples)
            """
            #======================================================================
    
            # after_train_step()
            
            if use_amp:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            
            #======================================================================
            # attention here! 
            """
            self._accelerator.backward(loss)
            """
            #======================================================================
        
            # update gradients every accum_grad_iters iterations
            
            if (i + 1) % accum_grad_iters == 0:
                if use_amp:
                    scaler.step(optimizer)
                    scaler.update()                     
                else:    
                    optimizer.step()
                optimizer.zero_grad()
            
            
            # Update gradients every accum_grad_iters iterations
            #======================================================================
            # attention here! 
            # Update gradients every accum_grad_iters iterations
            """
            if (i + 1) % accum_grad_iters == 0:
                self._accelerator.clip_grad_norm_(self._model.parameters(), max_norm=1.0)  # Optional: Gradient clipping
            
                # Step the optimizer
                self.optimizer.step()
                self.optimizer.zero_grad()  # Reset gradients
            """
            #======================================================================
            
    
            metric_logger.update(loss=loss.item())
            metric_logger.update(lr=optimizer.param_groups[0]["lr"])
            torch.cuda.empty_cache()
    
        print("Averaged stats: " + str(metric_logger.global_avg()))
        
        return {
            k: "{:.3f}".format(meter.global_avg)
            for k, meter in metric_logger.meters.items()
        }


    def train_step(self, model, samples):
        loss = model(samples)["loss"]
        return loss


    def eval_epoch(self):
        self._model.eval()
        return self.evaluation(model=self._model, data_loader=self.test_loader, iters_per_epoch=self.training_config.eval_iters_per_epoch)


    def valid_step(self, model, samples):
        outputs= model.generate_for_samples(samples)
        return outputs

    
    def log_and_print_metrics(self, cur_epoch, val_log):
        """
        Saves validation metrics to a JSON file and prints them dynamically using key-value pairs.

        Args:
            cur_epoch (int): The current epoch number.
            val_log (dict): A dictionary containing validation metrics such as loss, auc, uauc, etc.

        Raises:
            AssertionError: If val_log is empty.
        """
        # Ensure val_log is not empty
        assert val_log, "Validation log is empty. Cannot log metrics."

        # Prepare the metrics message dynamically
        metrics_message = [f"Epoch {cur_epoch} Metrics:"]
        for key, value in val_log.items():
            # Handle float formatting for numeric values
            if isinstance(value, (float, int)):
                metrics_message.append(f"  {key.capitalize()}: {value:.4f}")
            else:
                metrics_message.append(f"  {key.capitalize()}: {value}")

        # Join all parts into a single message
        message = "\n".join(metrics_message)

        # Save metrics to JSON
        log_data = {"epoch": cur_epoch, **val_log}  # Include the current epoch in the JSON entry
        try:
            with open(self.json_log_path, "a") as f:
                f.write(json.dumps(log_data) + "\n")  # Write each epoch's metrics on a new line
        except Exception as e:
            print(f"Error saving metrics to JSON: {e}")

        # Print the message
        print(message)


    def save_checkpoint(self, epoch, is_best=False):
        """
        Saves the model and optimizer states along with the current epoch to a checkpoint file.

        Args:
            epoch (int): The current epoch to be saved.
            is_regular_save (bool): Whether this save is a regular interval save or a metric-improvement save.
        """
        checkpoint_path = checkpoint_dir + "checkpoint_{}.pth".format("best" if is_best else epoch)

        param_grad_dic = {
            k: v.requires_grad for (k, v) in self._model.named_parameters()
        }

        state_dict = self._model.state_dict()
        for k in list(state_dict.keys()):
            if k in param_grad_dic.keys() and not param_grad_dic[k]:
                # delete parameters that do not require gradient
                del state_dict[k]
        
        checkpoint = {
            "model_state_dict": state_dict,
            "optimizer_state_dict": self.optimizer.state_dict(),
            "scaler": self.scaler.state_dict() if self.scaler else None,
            "epoch": epoch,
        }
        torch.save(checkpoint, checkpoint_path)
        print(f"checkpoint saved at {checkpoint_path}")



    def load_checkpoint(self, checkpoint_path=None):
        """
        Loads a checkpoint file to resume training.
    
        Args:
            checkpoint_path (str, optional): Path to the specific checkpoint file. If None, loads the latest checkpoint.
            checkpoint_dir (str, optional): Directory to search for the latest checkpoint if no path is provided.
    
        Returns:
            None
        """
        def _load_from_file(path):
            """Helper function to load checkpoint from a given file path."""
            checkpoint = torch.load(path, map_location=self._device)
            self._model.load_state_dict(checkpoint["model_state_dict"], strict=False)
            self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            self.scaler.load_state_dict(checkpoint["scaler"])
            self.start_epoch = checkpoint["epoch"] + 1
            print(f"Loaded checkpoint from {path}, resuming at epoch {self.start_epoch}")
    
        
        if checkpoint_path:
            # Check if the specified checkpoint exists
            if os.path.isfile(checkpoint_path):
                _load_from_file(checkpoint_path)
            else:
                print(f"No checkpoint found at {checkpoint_path}. Starting training from scratch.")
        else:
            # Load the best checkpoint 
            best_checkpoint = checkpoint_dir + "checkpoint_best.pth"
            if os.path.isfile(best_checkpoint):
                _load_from_file(best_checkpoint)
            else: 
                print("No model loaded")


    
    @torch.no_grad()
    def evaluation(self, model, data_loader, iters_per_epoch, use_auc=True):
        """
        Evaluates the model on the given data loader and computes metrics including AUC, accuracy, precision, recall, F1-Score, and log loss.
    
        Args:
            model (torch.nn.Module): The model to evaluate.
            data_loader (DataLoader): DataLoader providing the evaluation data.
            use_auc (bool): Whether to compute AUC metrics.
    
        Returns:
            dict: Evaluation results containing various metrics.
        """
        # Initialize metric loggers
        metric_logger = MetricLogger(delimiter="  ")
        metric_logger.add_meter("loss", SmoothedValue(window_size=1, fmt="{value:.4f}"))
        metric_logger.add_meter("acc", SmoothedValue(window_size=1, fmt="{value:.4f}"))
        
        if use_auc:
            auc_logger = MetricLogger(delimiter="  ")
            auc_logger.add_meter("auc", SmoothedValue(window_size=1, fmt="{value:.4f}"))
    
        header = "Evaluation"
        print_freq = 200  # Frequency of logging
    
        # Initialize results and labels
        results_logits = []
        labels = []
        users = []
    
        for i in metric_logger.log_every(range(iters_per_epoch), self.log_freq, header):
            # Move samples to CUDA if available
            samples = next(data_loader)
            samples = move_sample_to_cuda(samples)
    
            # Perform validation step
            eval_output = self.valid_step(model=model, samples=samples)
    
            # Log loss
            if "loss" in eval_output:
                metric_logger.update(loss=eval_output["loss"].item())
    
            # Log accuracy and logits for AUC
            if "logits" in eval_output:
                logits = eval_output["logits"]
                
                labels.extend(samples["label"].detach().cpu().numpy())
                users.extend(samples["UserID"].detach().cpu().numpy())

                #======================================================================
                #gather data from multi-gpus (used when in ddp mode)
                # logits = self._accelerator.gather_for_metrics(logits)
                # labels = self._accelerator.gather_for_metrics(labels)
                #======================================================================
                
                results_logits.extend(logits.detach().cpu().numpy())
    
                # Compute accuracy
                probabilities = torch.sigmoid(logits)
                predictions = (probabilities > 0.5).int()
                acc = (predictions == samples["label"]).float().mean()
                metric_logger.update(acc=acc.item())
            else:
                metric_logger.update(acc=0.0)
    
            torch.cuda.empty_cache()

        # Compute additional metrics if applicable
        if results_logits:
            results_logits_tensor = torch.tensor(results_logits).contiguous()
            labels_tensor = torch.tensor(labels).contiguous()
            users_tensor = torch.tensor(users).contiguous()
    
            # Apply sigmoid to convert logits to probabilities
            probabilities = torch.sigmoid(results_logits_tensor)
    
            # Convert tensors to numpy arrays for compatibility with sklearn
            probabilities_np = probabilities.numpy()
            labels_np = labels_tensor.numpy()
            users_np = users_tensor.numpy()
            
            # AUC
            auc = roc_auc_score(labels_np, probabilities_np) if use_auc else None
            uauc, _, _ = compute_user_auc(users_np, probabilities_np, labels_np) if use_auc else None
    
            # Precision, Recall, F1-Score
            predictions_np = (probabilities_np > 0.5).astype(int)
            precision = precision_score(labels_np, predictions_np, zero_division=0)
            recall = recall_score(labels_np, predictions_np, zero_division=0)
            f1 = f1_score(labels_np, predictions_np, zero_division=0)
    
            # Log Loss
            log_loss_value = log_loss(labels_np, probabilities_np)
    
            # Aggregate results
            results = {
                "loss": metric_logger.meters["loss"].global_avg,
                "acc": metric_logger.meters["acc"].global_avg,
                "auc": auc,
                "uauc": uauc,
                "precision": precision,
                "recall": recall,
                "f1_score": f1,
                "log_loss": log_loss_value,
            }
        else:
            results = {
                "loss": metric_logger.meters["loss"].global_avg,
                "accuracy": metric_logger.meters["acc"].global_avg,
            }
    
        return results


    def setup_seeds(self):
        seed = self.training_config.seed
    
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
    
        cudnn.benchmark = False
        cudnn.deterministic = True


    def train(self):
        start_time = time.time()
        best_epoch = 0
        self.setup_seeds()
        self.load_checkpoint()
        #======================================================================
        # initialize accelerator and auto move data/model to accelerator.device
        # self.prepare_accelerator()
        #======================================================================
    
        for cur_epoch in range(self.start_epoch, self.max_epoch):
            print(f"Epoch {cur_epoch + 1}/{self.max_epoch}")
    
            # Training phase
            print("Training Phase")
            train_stats = self.train_epoch(cur_epoch)
    
            # Evaluation phase
            print("Evaluation Phase")
            val_log = self.eval_epoch()
            torch.cuda.empty_cache()

            # TODO: improve this part of code to make it a separate procedure
            if val_log is not None:
                # Ensure required metrics are present
                required_metrics = ["loss", "auc", "uauc", "acc", "precision", "recall", "f1_score", "log_loss"]
                for metric in required_metrics:
                    assert metric in val_log, f"Metric '{metric}' not found in validation log."
    
                self.log_and_print_metrics(cur_epoch, val_log)

                self.early_stopper.update(val_log)

                if cur_epoch % self.save_frequency == 0:
                    self.save_checkpoint(cur_epoch)

                if self.early_stopper.should_stop():
                    best_epoch = cur_epoch
                    print("Early stopping triggered. Best Epoch: ", best_epoch)
                    break
    
        # Log total training time
        total_time = time.time() - start_time
        total_time_str = str(datetime.timedelta(seconds=int(total_time)))
        print(f"Training completed in {total_time_str}.")
        print("Training complete. Best metrics:", self.early_stopper.best_metric)

In [29]:
data_loaders = create_dataloaders(datasets)

In [30]:
runner = Runner(training_cfg, model, data_loaders)

number of trainable parameters: 5127168


## Training Phase

In [ ]:
runner.train()

No model loaded
Epoch 1/300
Training Phase
Start training epoch 0, 50 iters per inner epoch.
Train: data epoch: [0]  [ 0/50]  eta: 0:07:22  lr: 0.000010  loss: 5.2695  time: 8.8519  data: 0.0000  max mem: 5873


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


Train: data epoch: [0]  [49/50]  eta: 0:00:01  lr: 0.000253  loss: 0.7419  time: 1.5670  data: 0.0000  max mem: 9366
Train: data epoch: [0] Total time: 0:01:23 (1.6747 s / it)
Averaged stats: lr: 0.000131  loss: 1.898101
Evaluation Phase
Evaluation  [ 0/50]  eta: 0:02:24  loss: 0.6821  acc: 0.6250  time: 2.8812  data: 0.0000  max mem: 9366
Evaluation  [49/50]  eta: 0:00:02  loss: 0.7579  acc: 0.3281  time: 2.7788  data: 0.0000  max mem: 9533
Evaluation Total time: 0:02:16 (2.7297 s / it)
Users with only one interaction: 14
Valid users: 89, Invalid users: 10
Epoch 0 Metrics:
  Loss: 0.6909
  Acc: 0.5509
  Auc: 0.4878
  Uauc: 0.5035
  Precision: 0.5640
  Recall: 0.9009
  F1_score: 0.6937
  Log_loss: 0.6909
checkpoint saved at /kaggle/working/checkpoint_0.pth
Epoch 2/300
Training Phase
Start training epoch 1, 50 iters per inner epoch.
Train: data epoch: [1]  [ 0/50]  eta: 0:01:20  lr: 0.000010  loss: 0.7021  time: 1.6083  data: 0.0000  max mem: 9533
Train: data epoch: [1]  [49/50]  eta: 0

In [ ]:
from accelerate import notebook_launcher
notebook_launcher(runner.train, num_processes=8)